In [0]:
%run ../utils/init.py

In [0]:


df_bronze = spark.read.format("delta").load(f"{BRONZE}/dvf_transactions/")
print(f"Lignes bronze : {df_bronze.count()}")
df_bronze.printSchema()

In [0]:
from pyspark.sql.functions import to_date, year, month

df_silver = (df_bronze
    # Garder uniquement Appartement et Maison
    .filter(F.col("type_local").isin(["Appartement", "Maison"]))
    # Supprimer sans coordonnées
    .filter(F.col("latitude").isNotNull() & F.col("longitude").isNotNull())
    # Supprimer prix/m² aberrants
    .filter(F.col("prix_m2").isNull() | F.col("prix_m2").between(500, 50_000))
    # Date propre
    .withColumn("date_mutation", to_date("date_mutation", "yyyy-MM-dd"))
    .withColumn("annee", year("date_mutation"))
    .withColumn("mois", month("date_mutation"))
    # Typage
    .withColumn("rooms_count", F.col("nombre_pieces_principales").cast("int"))
    # Renommage snake_case anglais
    .withColumnRenamed("valeur_fonciere", "price")
    .withColumnRenamed("surface_reelle_bati", "building_area")
    .withColumnRenamed("surface_terrain", "land_area")
    .withColumnRenamed("type_local", "property_type")
    .withColumnRenamed("nom_commune", "city")
    .withColumnRenamed("prix_m2", "price_per_m2")
    # Déduplication
    .dropDuplicates(["id_mutation"])
    # Sélection finale
    .select(
        "id_mutation", "date_mutation", "annee", "mois",
        "price", "price_per_m2", "property_type",
        "building_area", "land_area", "rooms_count",
        "code_commune", "city", "code_departement",
        "longitude", "latitude"
    )
)

print(f"Lignes silver : {df_silver.count()}")
df_silver.show(3)

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("annee", "code_departement") \
    .save(f"{SILVER}/dvf_transactions/")

print("✅ Silver DVF transactions écrit")